# PySpark — Data Engineering Interview Practice (80 questions)

**8 categories × 10 questions**. Run the setup cell first. DataFrames are loaded from the local `data/` folder. Use **PySpark DataFrame API** unless a question asks otherwise. Every exercise has its own independent code cell. For practice, use notebook 02; for reference, use notebook 03.

Requires Python, Java 17 or 21 compatible with your installed Spark, and `pyspark` (install with `%pip install pyspark`). For Windows, configure Java/JAVA_HOME and a compatible PySpark version. This is **local Spark**, not a Databricks workspace.


> **Realistic Dataset v2:** Shared across Python, Pandas, and PySpark. Existing columns retained. Synthetic data includes intentional dirty records and malformed JSON for interview exercises.


In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F, Window, types as T
from IPython.display import display

# Run from the extracted project directory in VS Code/Jupyter.
BASE = Path.cwd()
if not (BASE / 'data').exists():
    raise FileNotFoundError('Open the notebook from the pyspark_de_practice directory (data/ required).')
spark = (SparkSession.builder.master('local[*]').appName('PySpark DE Interview Practice')
         .config('spark.sql.session.timeZone','UTC')
         .config('spark.sql.shuffle.partitions','8')
         .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')

csv_tables = ['customers','products','orders','order_items','transactions','pipeline_logs','cdc_changes']
for name in csv_tables:
    globals()[name] = (spark.read.option('header',True).option('inferSchema',True)
                       .csv(str(BASE / 'data' / (name+'.csv'))))
# Explicit nested JSON input (avoid treating nested properties as strings).
events = spark.read.json(str(BASE / 'data' / 'events.jsonl'))
# Corrupt records are preserved in a string column; explicit schema is important.
malformed_events = (spark.read.schema(T.StructType([
    T.StructField('event_id',T.LongType()),
    T.StructField('event_type',T.StringType()),
    T.StructField('_corrupt_record',T.StringType())
])).option('mode','PERMISSIVE').option('columnNameOfCorruptRecord','_corrupt_record')
.json(str(BASE / 'data' / 'malformed_events.jsonl')))
print('Spark version:',spark.version)
for name in csv_tables+['events']:
    print(f'{name}: {globals()[name].count()} rows')


## 1. DataFrame Fundamentals


### Q01 — Preview orders

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Show the first 10 orders.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = orders.limit(10)
result.show(20, truncate=False)


### Q02 — Project columns

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Select order_id, customer_id, amount.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = orders.select('order_id','customer_id','amount')
result.show(20, truncate=False)


### Q03 — Completed orders

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Keep completed orders.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = orders.filter(F.col('status') == 'completed')
result.show(20, truncate=False)


### Q04 — High value

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Keep orders >= 300.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = orders.filter(F.col('amount') >= 300)
result.show(20, truncate=False)


### Q05 — Rename field

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Rename amount to order_amount.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = orders.withColumnRenamed('amount','order_amount')
result.show(20, truncate=False)


### Q06 — Derived CAD amount

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Multiply amount by illustrative rate 1.35.

**Requirements:**
- Use numeric calculations and keep monetary comparisons precise enough for cents.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = orders.withColumn('amount_cad',F.col('amount')*F.lit(1.35))
result.show(20, truncate=False)


### Q07 — Distinct statuses

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Return unique order statuses.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = orders.select('status').distinct()
result.show(20, truncate=False)


### Q08 — Missing customer IDs

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Count order records where customer_id is NULL. Return exactly one row with the column missing_customer_ids; a valid result can be zero.

**Requirements:**
- Include missing or malformed values in the analysis rather than silently discarding them; report the relevant rows or counts as requested.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = orders.agg(F.sum(F.when(F.col('customer_id').isNull(),1).otherwise(0)).alias('missing_customer_ids'))
result.show(20, truncate=False)


### Q09 — Add literal source

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Add source_system = ecommerce.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = orders.withColumn('source_system',F.lit('ecommerce'))
result.show(20, truncate=False)


### Q10 — Conditional band

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Add high/standard value_band at 250.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = orders.withColumn('value_band',F.when(F.col('amount')>=250,'high').otherwise('standard'))
result.show(20, truncate=False)


## 2. Filtering, Sorting & SQL Expressions


### Q11 — Multiple predicates

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Completed orders with amount >= 200.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = orders.filter((F.col('status')=='completed') & (F.col('amount')>=200))
result.show(20, truncate=False)


### Q12 — IN predicate

**Interview scenario:** You are building a customer master-data platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Customers in CA or US.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `customers`: `customer_id`, `name`, `email`, `segment`, `country`, `signup_date`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = customers.filter(F.col('country').isin('CA','US'))
result.show(20, truncate=False)


### Q13 — Exclude cancelled

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Keep non-cancelled orders.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = orders.filter(F.col('status') != 'cancelled')
result.show(20, truncate=False)


### Q14 — Top five

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Five largest orders with stable order_id tie-break.

**Requirements:**
- Define a deterministic ordering or tie-breaker where records can have equal values; preserve the required row granularity.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = orders.orderBy(F.desc('amount'),F.asc('order_id')).limit(5)
result.show(20, truncate=False)


### Q15 — Multi-column sort

**Interview scenario:** You are building a customer master-data platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Sort customers by country and signup_date.

**Requirements:**
- Define a deterministic ordering or tie-breaker where records can have equal values; preserve the required row granularity.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `customers`: `customer_id`, `name`, `email`, `segment`, `country`, `signup_date`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = customers.orderBy('country','signup_date')
result.show(20, truncate=False)


### Q16 — Case insensitive

**Interview scenario:** You are building a customer master-data platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Find enterprise segment customers.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `customers`: `customer_id`, `name`, `email`, `segment`, `country`, `signup_date`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = customers.filter(F.lower(F.col('segment'))=='enterprise')
result.show(20, truncate=False)


### Q17 — Log keyword match

**Interview scenario:** You are building an ETL observability system. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Find messages containing error or fail.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `pipeline_logs`: `log_id`, `pipeline`, `level`, `duration_ms`, `timestamp`, `message`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = pipeline_logs.filter(F.lower(F.col('message')).rlike('error|fail'))
result.show(20, truncate=False)


### Q18 — Range predicate

**Interview scenario:** You are building a multi-currency payment ingestion service. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Transactions between 100 and 500 inclusive.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `transactions`: `transaction_id`, `customer_id`, `amount`, `currency`, `transaction_ts`, `type`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = transactions.filter(F.col('amount').between(100,500))
result.show(20, truncate=False)


### Q19 — Null-safe fallback

**Interview scenario:** You are building a customer master-data platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Create cleaned_email replacing null with unknown.

**Requirements:**
- Include missing or malformed values in the analysis rather than silently discarding them; report the relevant rows or counts as requested.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `customers`: `customer_id`, `name`, `email`, `segment`, `country`, `signup_date`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = customers.withColumn('cleaned_email',F.coalesce(F.col('email'),F.lit('unknown')))
result.show(20, truncate=False)


### Q20 — SQL expression

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Create gross_amount as amount * 1.05 using expr.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = orders.withColumn('gross_amount',F.expr('amount * 1.05'))
result.show(20, truncate=False)


## 3. Aggregations & GroupBy


### Q21 — Order status counts

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Count orders by status.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = orders.groupBy('status').count()
result.show(20, truncate=False)


### Q22 — Revenue by status

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Sum amount by status.

**Requirements:**
- Use numeric calculations and keep monetary comparisons precise enough for cents.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = orders.groupBy('status').agg(F.sum('amount').alias('revenue'))
result.show(20, truncate=False)


### Q23 — Customer spend

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Total order amount by customer.

**Requirements:**
- Use numeric calculations and keep monetary comparisons precise enough for cents.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = orders.groupBy('customer_id').agg(F.sum('amount').alias('total_spend'))
result.show(20, truncate=False)


### Q24 — Order KPIs

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Count, average, minimum and maximum amount by status.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = orders.groupBy('status').agg(F.count('*').alias('n'),F.avg('amount').alias('avg'),F.min('amount').alias('min'),F.max('amount').alias('max'))
result.show(20, truncate=False)


### Q25 — Segment distribution

**Interview scenario:** You are building a customer master-data platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Count customers by country and segment.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `customers`: `customer_id`, `name`, `email`, `segment`, `country`, `signup_date`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = customers.groupBy('country','segment').count()
result.show(20, truncate=False)


### Q26 — Completed revenue

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Completed-order revenue by customer.

**Requirements:**
- Use numeric calculations and keep monetary comparisons precise enough for cents.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = orders.filter(F.col('status')=='completed').groupBy('customer_id').agg(F.sum('amount').alias('completed_revenue'))
result.show(20, truncate=False)


### Q27 — Product units

**Interview scenario:** You are building an e-commerce order-line fact pipeline. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Total quantity by product_id.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `order_items`: `item_id`, `order_id`, `product_id`, `quantity`, `unit_price`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = order_items.groupBy('product_id').agg(F.sum('quantity').alias('units'))
result.show(20, truncate=False)


### Q28 — Line sales

**Interview scenario:** You are building an e-commerce order-line fact pipeline. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Quantity times unit_price aggregated by product.

**Requirements:**
- Use numeric calculations and keep monetary comparisons precise enough for cents.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `order_items`: `item_id`, `order_id`, `product_id`, `quantity`, `unit_price`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = order_items.withColumn('line_total',F.col('quantity')*F.col('unit_price')).groupBy('product_id').agg(F.sum('line_total').alias('sales'))
result.show(20, truncate=False)


### Q29 — Conditional counts

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Count completed and cancelled orders per customer.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = orders.groupBy('customer_id').agg(F.sum(F.when(F.col('status')=='completed',1).otherwise(0)).alias('completed'),F.sum(F.when(F.col('status')=='cancelled',1).otherwise(0)).alias('cancelled'))
result.show(20, truncate=False)


### Q30 — Approx distinct

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Approximate distinct customers by status.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = orders.groupBy('status').agg(F.approx_count_distinct('customer_id').alias('approx_customers'))
result.show(20, truncate=False)


## 4. Joins & Data Modeling


### Q31 — Enrich orders

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Attach customer name and segment to orders.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`
- `customers`: `customer_id`, `name`, `email`, `segment`, `country`, `signup_date`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = orders.join(customers.select('customer_id','name','segment'),'customer_id','left')
result.show(20, truncate=False)


### Q32 — Customer order counts

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Preserve all customers, including zero orders.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `customers`: `customer_id`, `name`, `email`, `segment`, `country`, `signup_date`
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
counts=orders.groupBy('customer_id').count().withColumnRenamed('count','order_count');result=customers.join(counts,'customer_id','left').fillna({'order_count':0})
result.show(20, truncate=False)


### Q33 — Order line enrichment

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Attach order status and order_date to each line.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `order_items`: `item_id`, `order_id`, `product_id`, `quantity`, `unit_price`
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = order_items.join(orders.select('order_id','status','order_date'),'order_id','left')
result.show(20, truncate=False)


### Q34 — Product category enrichment

**Interview scenario:** You are building an e-commerce order-line fact pipeline. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Attach category to order items.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `order_items`: `item_id`, `order_id`, `product_id`, `quantity`, `unit_price`
- `products`: `product_id`, `sku`, `category`, `price`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = order_items.join(products.select('product_id','category'),'product_id','left')
result.show(20, truncate=False)


### Q35 — Category revenue

**Interview scenario:** You are building an e-commerce order-line fact pipeline. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Aggregate item revenue by product category.

**Requirements:**
- Use numeric calculations and keep monetary comparisons precise enough for cents.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `order_items`: `item_id`, `order_id`, `product_id`, `quantity`, `unit_price`
- `products`: `product_id`, `sku`, `category`, `price`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = order_items.join(products.select('product_id','category'),'product_id','left').groupBy('category').agg(F.sum(F.col('quantity')*F.col('unit_price')).alias('revenue'))
result.show(20, truncate=False)


### Q36 — Left anti join

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Find customers with no orders.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `customers`: `customer_id`, `name`, `email`, `segment`, `country`, `signup_date`
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = customers.join(orders.select('customer_id').distinct(),'customer_id','left_anti')
result.show(20, truncate=False)


### Q37 — Orphan orders

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Return orders whose non-null customer_id has no matching customer master record. A null customer_id is also an orphan. Return the original order columns.

**Requirements:**
- Include missing or malformed values in the analysis rather than silently discarding them; report the relevant rows or counts as requested.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`
- `customers`: `customer_id`, `name`, `email`, `segment`, `country`, `signup_date`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = orders.join(customers.select('customer_id'),'customer_id','left_anti')
result.show(20, truncate=False)


### Q38 — Broadcast dimension

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Join orders to broadcasted customers dimension.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`
- `customers`: `customer_id`, `name`, `email`, `segment`, `country`, `signup_date`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = orders.join(F.broadcast(customers.select('customer_id','segment')),'customer_id','left')
result.show(20, truncate=False)


### Q39 — Semi join

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Return customers who have at least one order.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `customers`: `customer_id`, `name`, `email`, `segment`, `country`, `signup_date`
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = customers.join(orders.select('customer_id').distinct(),'customer_id','left_semi')
result.show(20, truncate=False)


### Q40 — Join validation

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Report orders row count before and after joining customers.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`
- `customers`: `customer_id`, `name`, `email`, `segment`, `country`, `signup_date`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
joined=orders.join(customers.select('customer_id','name'),'customer_id','left');result=spark.createDataFrame([(orders.count(),joined.count())],['before','after'])
result.show(20, truncate=False)


## 5. Window Functions & Top-N


### Q41 — Latest order

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** For each customer_id, select exactly one most recent order. Sort by order_date descending and break equal-date ties with the larger order_id. Return original order columns.

**Requirements:**
- Define a deterministic ordering or tie-breaker where records can have equal values; preserve the required row granularity.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
w=Window.partitionBy('customer_id').orderBy(F.desc('order_date'),F.desc('order_id'));result=orders.withColumn('rn',F.row_number().over(w)).filter('rn = 1').drop('rn')
result.show(20, truncate=False)


### Q42 — Customer running spend

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** For every order, calculate cumulative order amount for that customer through the current order. Order by order_date ascending, then order_id ascending. Keep every original order row.

**Requirements:**
- Partition by the business key and order chronologically; do not mix records from different customers or sessions.
- Use numeric calculations and keep monetary comparisons precise enough for cents.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
w=Window.partitionBy('customer_id').orderBy('order_date','order_id').rowsBetween(Window.unboundedPreceding,Window.currentRow);result=orders.withColumn('running_spend',F.sum('amount').over(w))
result.show(20, truncate=False)


### Q43 — Previous amount

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Previous order amount per customer.

**Requirements:**
- Partition by the business key and order chronologically; do not mix records from different customers or sessions.
- Use numeric calculations and keep monetary comparisons precise enough for cents.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
w=Window.partitionBy('customer_id').orderBy('order_date','order_id');result=orders.withColumn('previous_amount',F.lag('amount').over(w))
result.show(20, truncate=False)


### Q44 — Next order date

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Next order date per customer.

**Requirements:**
- Partition by the business key and order chronologically; do not mix records from different customers or sessions.
- Handle date/time fields as dates or timestamps (UTC where applicable), not as arbitrary text.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
w=Window.partitionBy('customer_id').orderBy('order_date','order_id');result=orders.withColumn('next_order_date',F.lead('order_date').over(w))
result.show(20, truncate=False)


### Q45 — Rank by amount

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Dense rank orders by amount within status.

**Requirements:**
- Define a deterministic ordering or tie-breaker where records can have equal values; preserve the required row granularity.
- Use numeric calculations and keep monetary comparisons precise enough for cents.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
w=Window.partitionBy('status').orderBy(F.desc('amount'));result=orders.withColumn('dense_rank',F.dense_rank().over(w))
result.show(20, truncate=False)


### Q46 — Top three customers

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Top three customers by spend in each segment.

**Requirements:**
- Define a deterministic ordering or tie-breaker where records can have equal values; preserve the required row granularity.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `customers`: `customer_id`, `name`, `email`, `segment`, `country`, `signup_date`
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
spend=orders.groupBy('customer_id').agg(F.sum('amount').alias('spend'));x=customers.join(spend,'customer_id','left').fillna({'spend':0});w=Window.partitionBy('segment').orderBy(F.desc('spend'),F.asc('customer_id'));result=x.withColumn('rn',F.row_number().over(w)).filter('rn <= 3')
result.show(20, truncate=False)


### Q47 — Customer order count

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Annotate every order with the total number of orders placed by its customer. Preserve all order rows (do not GROUP BY and collapse them).

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
w=Window.partitionBy('customer_id');result=orders.withColumn('customer_order_count',F.count('*').over(w))
result.show(20, truncate=False)


### Q48 — Spend share

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Percent of customer total represented by each order.

**Requirements:**
- Use numeric calculations and keep monetary comparisons precise enough for cents.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
w=Window.partitionBy('customer_id');result=orders.withColumn('share_pct',F.when(F.sum('amount').over(w)!=0,F.col('amount')/F.sum('amount').over(w)*100))
result.show(20, truncate=False)


### Q49 — Deduplicate CDC

**Interview scenario:** You are building a change-data-capture replication pipeline. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Latest change per customer, changed_at/change_id tie-break.

**Requirements:**
- Define a deterministic ordering or tie-breaker where records can have equal values; preserve the required row granularity.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `cdc_changes`: `change_id`, `customer_id`, `operation`, `changed_at`, `email`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
w=Window.partitionBy('customer_id').orderBy(F.desc('changed_at'),F.desc('change_id'));result=cdc_changes.withColumn('rn',F.row_number().over(w)).filter('rn = 1').drop('rn')
result.show(20, truncate=False)


### Q50 — Days since prior order

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Days between consecutive orders per customer.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
w=Window.partitionBy('customer_id').orderBy('order_date','order_id');result=orders.withColumn('days_since_previous',F.datediff(F.to_date('order_date'),F.to_date(F.lag('order_date').over(w))))
result.show(20, truncate=False)


## 6. Dates, Events & Streaming Concepts


### Q51 — Parse timestamps

**Interview scenario:** You are building a multi-currency payment ingestion service. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Convert transaction_ts to Spark timestamp.

**Requirements:**
- Handle date/time fields as dates or timestamps (UTC where applicable), not as arbitrary text.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `transactions`: `transaction_id`, `customer_id`, `amount`, `currency`, `transaction_ts`, `type`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = transactions.withColumn('ts',F.to_timestamp('transaction_ts'))
result.show(20, truncate=False)


### Q52 — Monthly revenue

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Sum order amounts by calendar month.

**Requirements:**
- Handle date/time fields as dates or timestamps (UTC where applicable), not as arbitrary text.
- Use numeric calculations and keep monetary comparisons precise enough for cents.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = orders.withColumn('month',F.date_trunc('month',F.to_timestamp('order_date'))).groupBy('month').agg(F.sum('amount').alias('revenue'))
result.show(20, truncate=False)


### Q53 — Daily transaction count

**Interview scenario:** You are building a multi-currency payment ingestion service. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Count transactions per UTC calendar day.

**Requirements:**
- Handle date/time fields as dates or timestamps (UTC where applicable), not as arbitrary text.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `transactions`: `transaction_id`, `customer_id`, `amount`, `currency`, `transaction_ts`, `type`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = transactions.withColumn('day',F.to_date('transaction_ts')).groupBy('day').count()
result.show(20, truncate=False)


### Q54 — Event daily active users

**Interview scenario:** You are building a user-activity telemetry pipeline. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Count distinct event users per date.

**Requirements:**
- Handle date/time fields as dates or timestamps (UTC where applicable), not as arbitrary text.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `events`: `event_id`, `user_id`, `event_type`, `event_ts`, `session_id`, `properties`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = events.withColumn('day',F.to_date('event_ts')).groupBy('day').agg(F.countDistinct('user_id').alias('dau'))
result.show(20, truncate=False)


### Q55 — Session duration

**Interview scenario:** You are building a user-activity telemetry pipeline. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Compute min/max event timestamps and session duration.

**Requirements:**
- Partition by the business key and order chronologically; do not mix records from different customers or sessions.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `events`: `event_id`, `user_id`, `event_type`, `event_ts`, `session_id`, `properties`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
from pyspark.sql import functions as F

# Convert ISO 8601 strings to timestamps
events_fixed = events.withColumn(
    "event_ts",
    F.to_timestamp("event_ts", "yyyy-MM-dd'T'HH:mm:ssX")
)

# Calculate session boundaries
result = (
    events_fixed
    .groupBy("session_id")
    .agg(
        F.min("event_ts").alias("start"),
        F.max("event_ts").alias("end"),
        F.count("*").alias("events")
    )
    .withColumn(
        "duration_seconds",
        F.unix_timestamp("end") - F.unix_timestamp("start")
    )
)

result.show(20, truncate=False)

### Q56 — Watermark filter

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Select orders after 2025-09-01.

**Requirements:**
- Handle date/time fields as dates or timestamps (UTC where applicable), not as arbitrary text.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = orders.filter(F.to_date('order_date')>F.lit('2025-09-01').cast('date'))
result.show(20, truncate=False)


### Q57 — Hour buckets

**Interview scenario:** You are building a user-activity telemetry pipeline. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Count events per hour.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `events`: `event_id`, `user_id`, `event_type`, `event_ts`, `session_id`, `properties`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = events.groupBy(F.date_trunc('hour',F.to_timestamp('event_ts')).alias('hour')).count()
result.show(20, truncate=False)


### Q58 — Event-type distribution

**Interview scenario:** You are building a user-activity telemetry pipeline. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Count events by event_type.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `events`: `event_id`, `user_id`, `event_type`, `event_ts`, `session_id`, `properties`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = events.groupBy('event_type').count()
result.show(20, truncate=False)


### Q59 — Pipeline daily failures

**Interview scenario:** You are building an ETL observability system. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Count ERROR logs per UTC day and pipeline.

**Requirements:**
- Handle date/time fields as dates or timestamps (UTC where applicable), not as arbitrary text.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `pipeline_logs`: `log_id`, `pipeline`, `level`, `duration_ms`, `timestamp`, `message`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = pipeline_logs.withColumn('day',F.to_date('timestamp')).filter(F.upper('level')=='ERROR').groupBy('pipeline','day').count()
result.show(20, truncate=False)


### Q60 — Event time windows

**Interview scenario:** You are building a user-activity telemetry pipeline. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Count events in 1-hour tumbling windows (batch DataFrame; same window expression as streaming).

**Requirements:**
- Partition by the business key and order chronologically; do not mix records from different customers or sessions.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `events`: `event_id`, `user_id`, `event_type`, `event_ts`, `session_id`, `properties`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = events.withColumn('ts',F.to_timestamp('event_ts')).groupBy(F.window('ts','1 hour'),'event_type').count()
result.show(20, truncate=False)


## 7. Data Quality, CDC & Nested JSON


### Q61 — Normalize email

**Interview scenario:** You are building a customer master-data platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Lowercase and trim customer emails.

**Requirements:**
- Account for null and whitespace-only emails; normalize case/whitespace when matching or detecting duplicates.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `customers`: `customer_id`, `name`, `email`, `segment`, `country`, `signup_date`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = customers.withColumn('email_normalized',F.lower(F.trim('email')))
result.show(20, truncate=False)


### Q62 — Blank emails

**Interview scenario:** You are building a customer master-data platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Return customers with null or blank email.

**Requirements:**
- Include missing or malformed values in the analysis rather than silently discarding them; report the relevant rows or counts as requested.
- Account for null and whitespace-only emails; normalize case/whitespace when matching or detecting duplicates.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `customers`: `customer_id`, `name`, `email`, `segment`, `country`, `signup_date`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = customers.filter(F.col('email').isNull() | (F.trim('email')==''))
result.show(20, truncate=False)


### Q63 — Invalid transaction amounts

**Interview scenario:** You are building a multi-currency payment ingestion service. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Return null or non-positive transaction amounts.

**Requirements:**
- Include missing or malformed values in the analysis rather than silently discarding them; report the relevant rows or counts as requested.
- Use numeric calculations and keep monetary comparisons precise enough for cents.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `transactions`: `transaction_id`, `customer_id`, `amount`, `currency`, `transaction_ts`, `type`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = transactions.filter(F.col('amount').isNull() | (F.col('amount')<=0))
result.show(20, truncate=False)


### Q64 — Duplicate customer emails

**Interview scenario:** You are building a customer master-data platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Find repeated normalized email values.

**Requirements:**
- Define a deterministic ordering or tie-breaker where records can have equal values; preserve the required row granularity.
- Account for null and whitespace-only emails; normalize case/whitespace when matching or detecting duplicates.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `customers`: `customer_id`, `name`, `email`, `segment`, `country`, `signup_date`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
x=customers.withColumn('normalized',F.lower(F.trim('email')));dups=x.groupBy('normalized').count().filter('count > 1');result=x.join(dups.select('normalized'),'normalized','inner')
result.show(20, truncate=False)


### Q65 — CDC current state

**Interview scenario:** You are building a change-data-capture replication pipeline. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Keep latest CDC row per customer excluding DELETE/D.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `cdc_changes`: `change_id`, `customer_id`, `operation`, `changed_at`, `email`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
w=Window.partitionBy('customer_id').orderBy(F.desc('changed_at'),F.desc('change_id'));result=cdc_changes.withColumn('rn',F.row_number().over(w)).filter('rn = 1').filter(~F.upper('operation').isin('DELETE','D')).drop('rn')
result.show(20, truncate=False)


### Q66 — Schema type audit

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Return schema field names and data types for orders.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result=spark.createDataFrame([(x.name,x.dataType.simpleString(),x.nullable) for x in orders.schema.fields],['column','dtype','nullable'])
result.show(20, truncate=False)


### Q67 — Order-line reconciliation

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Compute SUM(quantity * unit_price) per order_id from order_items, then return orders where ABS(order amount - line total) > 0.01. Include the computed line_total.

**Requirements:**
- Calculate totals from the underlying line items and compare with the source amount; flag differences strictly greater than 0.01.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`
- `order_items`: `item_id`, `order_id`, `product_id`, `quantity`, `unit_price`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
lines=order_items.groupBy('order_id').agg(F.sum(F.col('quantity')*F.col('unit_price')).alias('line_total'));result=orders.join(lines,'order_id','left').filter(F.abs(F.col('amount')-F.coalesce(F.col('line_total'),F.lit(0)))>0.01)
result.show(20, truncate=False)


### Q68 — Missing key summary

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Return a two-row dataset containing the count of null order_id values in orders and the count of null customer_id values in customers. Columns: dataset and null_key_rows.

**Requirements:**
- Include missing or malformed values in the analysis rather than silently discarding them; report the relevant rows or counts as requested.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`
- `customers`: `customer_id`, `name`, `email`, `segment`, `country`, `signup_date`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result=spark.createDataFrame([('orders',orders.filter(F.col('order_id').isNull()).count()),('customers',customers.filter(F.col('customer_id').isNull()).count())],['dataset','null_key_rows'])
result.show(20, truncate=False)


### Q69 — Parse nested JSON

**Interview scenario:** You are building a user-activity telemetry pipeline. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Load events JSONL with inferred nested properties and select properties as a struct.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `events`: `event_id`, `user_id`, `event_type`, `event_ts`, `session_id`, `properties`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = events.select('event_id','event_type','properties')
result.show(20, truncate=False)


### Q70 — Malformed JSON quarantine

**Interview scenario:** You are building an event-ingestion quarantine pipeline. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Read malformed_events.jsonl using PERMISSIVE mode and flag corrupt raw records.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `malformed_events`: `event_id`, `event_type`, `_corrupt_record`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
malformed_events.cache().count();result=malformed_events.filter(F.col('_corrupt_record').isNotNull()).select('event_id','event_type','_corrupt_record')
result.show(20, truncate=False)


## 8. Production ETL & Spark Performance


### Q71 — Idempotent union

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Union overlapping order batches and deduplicate by order_id.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
a=orders.filter(F.col('order_id')<=100);b=orders.filter(F.col('order_id')<=150);result=a.unionByName(b).dropDuplicates(['order_id'])
result.show(20, truncate=False)


### Q72 — Partition distribution

**Interview scenario:** You are building a customer master-data platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Count rows by country to choose partition strategy.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `customers`: `customer_id`, `name`, `email`, `segment`, `country`, `signup_date`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = customers.groupBy('country').count().orderBy(F.desc('count'))
result.show(20, truncate=False)


### Q73 — Repartition by key

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Repartition orders into 8 partitions by customer_id.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = orders.repartition(8,'customer_id')
result.show(20, truncate=False)


### Q74 — Coalesce partitions

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Reduce orders to 2 partitions without a full shuffle.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = orders.coalesce(2)
result.show(20, truncate=False)


### Q75 — Cache reused data

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Cache orders, materialize cache, and select completed rows.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
cached=orders.cache();cached.count();result=cached.filter(F.col('status')=='completed')
result.show(20, truncate=False)


### Q76 — Skew inspection

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Show ten most frequent customer keys in orders.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = orders.groupBy('customer_id').count().orderBy(F.desc('count')).limit(10)
result.show(20, truncate=False)


### Q77 — Pipeline SLA

**Interview scenario:** You are building an ETL observability system. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Count runs over 60000ms by pipeline.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `pipeline_logs`: `log_id`, `pipeline`, `level`, `duration_ms`, `timestamp`, `message`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result = pipeline_logs.groupBy('pipeline').agg(F.count('*').alias('runs'),F.sum(F.when(F.col('duration_ms')>60000,1).otherwise(0)).alias('sla_breaches'))
result.show(20, truncate=False)


### Q78 — Currency normalization

**Interview scenario:** You are building a multi-currency payment ingestion service. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Normalize transactions using illustrative CAD rates.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `transactions`: `transaction_id`, `customer_id`, `amount`, `currency`, `transaction_ts`, `type`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
rates=F.create_map(F.lit('USD'),F.lit(1.35),F.lit('CAD'),F.lit(1.0),F.lit('EUR'),F.lit(1.48),F.lit('GBP'),F.lit(1.72));result=transactions.withColumn('amount_cad',F.col('amount')*F.element_at(rates,F.col('currency')))
result.show(20, truncate=False)


### Q79 — Data quality metrics

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Return exactly one row with total order count, number of orders missing customer_id, and number of orders with a negative amount. Use columns rows, missing_customer_ids, negative_amounts.

**Requirements:**
- Include missing or malformed values in the analysis rather than silently discarding them; report the relevant rows or counts as requested.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result=orders.agg(F.count('*').alias('rows'),F.sum(F.when(F.col('customer_id').isNull(),1).otherwise(0)).alias('missing_customer_ids'),F.sum(F.when(F.col('amount')<0,1).otherwise(0)).alias('negative_amounts'))
result.show(20, truncate=False)


### Q80 — Explain physical plan

**Interview scenario:** You are building an online retail order-processing platform. A teammate asks you to implement the following production-oriented transformation using PySpark DataFrame API.

**Task:** Print the physical plan of a grouped completed-order revenue query.

**Requirements:**
- Use the supplied columns as the source of truth, preserve the requested output granularity, and make the result reproducible.
- Implement with distributed DataFrame transformations; avoid collecting the entire dataset to the driver.

**Available datasets and columns:**
- `orders`: `order_id`, `customer_id`, `order_date`, `status`, `amount`

**Expected output:** Return or display the records, summary table, or derived values described in the task. Include the requested fields; when the task is a count, return the numeric count (zero is valid).

**Solution:**


In [ ]:
result=orders.filter(F.col('status')=='completed').groupBy('customer_id').agg(F.sum('amount').alias('revenue'));result.explain(mode='formatted')
result.show(20, truncate=False)
